# 03. Limpeza e Estruturação da Legislação
Transformamos as fontes auditadas em uma base rastreável, preservando o texto original, os metadados e a hierarquia detectada.

**Objetivos:**
1. Conferir os arquivos e hashes em relação à auditoria e à EDA.
2. Separar preâmbulos, títulos, artigos, parágrafos, incisos, alíneas e anexos.
3. Aplicar normalização conservadora sem descartar notas, links ou conteúdo jurídico.
4. Validar a cobertura do texto e exportar os resultados para os experimentos de chunking.

O parser é heurístico. Registros ambíguos ficam sinalizados para revisão. A presença de metadados não comprova vigência nem atualização jurídica.

## 1. Configuração do Ambiente e Localização dos Dados

In [11]:
import re
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import yaml

def find_raw_directory(start_path: Path) -> Path:
    # Retorna um Path válido ou interrompe a execução se a pasta não existir.
    for base in (start_path, *start_path.parents):
        candidate = base / 'data/raw/legislacao'
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f'Pasta data/raw/legislacao não encontrada a partir de {start_path}.'
    )


CWD = Path.cwd().resolve()
RAW_DIR: Path = find_raw_directory(CWD)

PROJECT_ROOT = RAW_DIR.parents[2]
PROCESSED_DIR = PROJECT_ROOT / 'data/processed/legislacao'
REPORT_DIR = PROJECT_ROOT / 'reports/legislacao/estruturacao'
AUDIT_DIR = PROJECT_ROOT / 'reports/legislacao/auditoria'
EDA_DIR = PROJECT_ROOT / 'reports/legislacao/eda'
EXECUTADO_EM = datetime.now(timezone.utc).isoformat()
PARSER_VERSION = '1.0.0'
print(f'✅ Diretório de entrada: {RAW_DIR}')

✅ Diretório de entrada: C:\Users\LAISA\OneDrive\Documentos\lex_clt.ai\data\raw\legislacao


## 2. Conferência das Fontes e dos Metadados
Os hashes são calculados sobre os bytes originais. As posições dos dispositivos são índices no texto decodificado em UTF-8, preservando suas quebras de linha, não posições em bytes. Valores de metadados ausentes não recebem informação jurídica presumida.

In [12]:
audit_path = AUDIT_DIR / 'auditoria_legislacao_resumo.csv'
eda_path = EDA_DIR / 'eda_rastreabilidade.csv'
for report_path in [audit_path, eda_path]:
    if not report_path.is_file():
        raise FileNotFoundError(f'Execute e exporte a etapa anterior: {report_path}')

df_audit = pd.read_csv(audit_path, dtype=str)
df_eda = pd.read_csv(eda_path, dtype=str)
if not {'caminho_relativo', 'sha256', 'status'}.issubset(df_audit.columns):
    raise ValueError('Relatório de auditoria incompatível. Execute novamente o notebook 01.')
if not {'caminho_relativo', 'sha256_eda', 'status_correspondencia'}.issubset(df_eda.columns):
    raise ValueError('Relatório da EDA incompatível. Execute novamente o notebook 02.')

source_records = []
source_texts = {}
metadata_records = []
for file_path in sorted(RAW_DIR.rglob('*.md')):
    if file_path.name.lower() == 'readme.md' or '.git' in file_path.relative_to(RAW_DIR).parts:
        continue
    caminho = file_path.relative_to(RAW_DIR).as_posix()
    raw_bytes = file_path.read_bytes()
    text = raw_bytes.decode('utf-8')
    source_texts[caminho] = text
    match = re.match(r'\A\ufeff?---[ \t]*\r?\n(.*?)\r?\n---[ \t]*(?:\r?\n|$)', text, re.DOTALL)
    metadata = {}
    error = ''
    if match:
        try:
            metadata = yaml.safe_load(match.group(1))
            if not isinstance(metadata, dict):
                raise ValueError('O YAML não contém um mapeamento.')
        except (yaml.YAMLError, ValueError) as e:
            metadata = {}
            error = str(e)
    else:
        error = 'Frontmatter ausente ou delimitadores não reconhecidos.'

    source_records.append({
        'caminho_relativo': caminho, 'nome_arquivo': file_path.name,
        'categoria': file_path.relative_to(RAW_DIR).parts[0],
        'sha256_atual': hashlib.sha256(raw_bytes).hexdigest(),
        'body_start': match.end() if match else 0,
        'caracteres_originais': len(text), 'erro_metadados': error
    })
    metadata_records.append({
        'caminho_relativo': caminho,
        'yaml_original': match.group(0) if match else '',
        'metadados_json': json.dumps(metadata, ensure_ascii=False, default=str),
        'identifier': metadata.get('identifier'), 'title': metadata.get('title'),
        'status_declarado': metadata.get('status'), 'source': metadata.get('source'),
        'date': str(metadata['date']) if metadata.get('date') is not None else None,
        'retrieved': str(metadata['retrieved']) if metadata.get('retrieved') is not None else None
    })

if not source_records:
    raise ValueError('Nenhuma norma encontrada. A execução foi interrompida.')
df_fontes = pd.DataFrame(source_records)
df_metadados = pd.DataFrame(metadata_records)
for dataframe in [df_audit, df_eda]:
    dataframe['caminho_relativo'] = dataframe['caminho_relativo'].str.replace('\\', '/', regex=False)
df_rastreabilidade = (
    df_fontes.merge(df_audit[['caminho_relativo', 'sha256', 'status']],
                    on='caminho_relativo', how='outer', validate='one_to_one')
    .merge(df_eda[['caminho_relativo', 'sha256_eda', 'status_correspondencia']],
           on='caminho_relativo', how='outer', validate='one_to_one')
)
df_rastreabilidade['integridade_ok'] = (
    df_rastreabilidade['sha256_atual'].notna() &
    df_rastreabilidade['sha256_atual'].eq(df_rastreabilidade['sha256']) &
    df_rastreabilidade['sha256_atual'].eq(df_rastreabilidade['sha256_eda']) &
    df_rastreabilidade['status'].eq('OK') &
    df_rastreabilidade['status_correspondencia'].eq('OK')
)
if not df_rastreabilidade['integridade_ok'].all():
    display(df_rastreabilidade[~df_rastreabilidade['integridade_ok']])
    raise ValueError('Fontes divergentes ou bloqueadas. Atualize a auditoria e a EDA antes de continuar.')

# Duplicidade exata é registrada, sem excluir automaticamente uma fonte jurídica.
df_duplicacoes = df_fontes[df_fontes['sha256_atual'].duplicated(keep=False)].copy()
df_duplicacoes['decisao'] = 'Preservado; revisar identidade e proveniência antes de excluir.'
print(f'• Normas com integridade confirmada: {len(df_fontes)}')
print(f'• Arquivos em grupos de duplicidade exata: {len(df_duplicacoes)}')
display(df_metadados[['caminho_relativo', 'status_declarado', 'source', 'retrieved']].head())

• Normas com integridade confirmada: 25
• Arquivos em grupos de duplicidade exata: 0


,caminho_relativo,status_declarado,source,retrieved
0,codigos/codigo-civil-lei-10406-2002.md,in_force,https://www.planalto.gov.br/ccivil_03/LEIS/200...,2026-10-12
1,codigos/codigo-de-processo-civil-lei-13105-201...,in_force,https://www.planalto.gov.br/ccivil_03/_Ato2015...,2026-10-12
2,codigos/consolidacao-das-leis-do-trabalho-clt-...,in_force,https://www.planalto.gov.br/ccivil_03/Decreto-...,2026-10-12
3,constituicao/constituicao-1988.md,in_force,https://www.planalto.gov.br/ccivil_03/constitu...,2026-10-12
4,leis/1949/lei-605-1949.md,repealed,https://www.planalto.gov.br/ccivil_03/leis/L06...,2026-10-12


## 3. Limpeza Conservadora e Registro das Transformações
Mantemos duas representações: texto original e texto normalizado. A limpeza normaliza apenas quebras de linha e espaços no fim das linhas. Não remove linhas vazias, indentação, destinos de links, marcas de revogação, notas ou tabelas HTML. O YAML é preservado na tabela de metadados.

In [13]:
def clean_legal_text(text):
    if not isinstance(text, str):
        raise TypeError('O conteúdo deve ser texto.')
    normalized = text.replace('\r\n', '\n').replace('\r', '\n')
    normalized = re.sub(r'[ \t]+(?=\n|$)', '', normalized)
    return normalized


def classify_line(line, inside_table=False):
    if inside_table:
        return None
    article = re.match(
        r'^[ \t]{0,3}(?:#{1,6}[ \t]+|\*\*)(Art\.?[ \t]*\d+(?:[.,]\d+)*(?:[º°])?(?:[-–—][A-Za-z]+)?)(?=[\s*.:]|$)',
        line, re.IGNORECASE
    )
    if article:
        return 'artigo', article.group(1)
    heading = re.match(r'^[ \t]{0,3}(?:#{1,6}\s+|\*\*)(.*)', line)
    if heading:
        label = heading.group(1).strip().strip('*').strip()
        for pattern, tipo in [
            (r'ANEXO\b', 'anexo'), (r'LIVRO\b', 'livro'),
            (r'PARTE\b', 'parte'), (r'T[ÍI]TULO\b', 'titulo'),
            (r'CAP[ÍI]TULO\b', 'capitulo'), (r'SUBSE[ÇC][ÃA]O\b', 'subsecao'),
            (r'SE[ÇC][ÃA]O\b', 'secao')
        ]:
            if re.match(pattern, label, re.IGNORECASE):
                return tipo, label
    prefix = r'^[ \t]*(?:\*\*)?'
    for pattern, tipo in [
        (r'(Parágrafo\s+único)\b', 'paragrafo'),
        (r'(§\s*\d+[º°]?)', 'paragrafo'),
        (r'([IVXLCDM]+)\s*[-–—)]', 'inciso'),
        (r'([a-z])\)', 'alinea')
    ]:
        match = re.match(prefix + pattern, line, re.IGNORECASE if tipo == 'paragrafo' else 0)
        if match:
            return tipo, match.group(1)
    return None

## 4. Parser Hierárquico e Preservação das Posições
Os intervalos são exclusivos: o texto de cada registro termina no início do próximo registro. O vínculo `parent_id` representa a hierarquia; o conteúdo completo de um artigo pode ser reconstruído agregando seus descendentes em ordem. O texto anterior ao primeiro dispositivo é preservado como preâmbulo. Linhas em tabelas HTML e blocos de código não são interpretadas como dispositivos. Um anexo encerra o contexto do artigo anterior.

In [14]:
def parse_norm_to_structured_records(caminho, text, body_start):
    events = []
    position = body_start
    in_table = False
    fence = None
    for line in text[body_start:].splitlines(keepends=True):
        stripped = line.lstrip()
        fence_match = re.match(r'(`{3,}|~{3,})', stripped)
        if fence_match:
            marker = fence_match.group(1)[0]
            fence = None if fence == marker else marker if fence is None else fence
            event = None
        else:
            if re.search(r'<table\b', line, re.IGNORECASE):
                in_table = True
            event = None if fence else classify_line(line, in_table)
            if re.search(r'</table\s*>', line, re.IGNORECASE):
                in_table = False
        if event:
            events.append((position, *event))
        position += len(line)

    if not events or events[0][0] > body_start:
        events.insert(0, (body_start, 'preambulo', 'Preâmbulo / estrutura inicial'))

    structural_levels = {'parte': 1, 'livro': 2, 'titulo': 3, 'capitulo': 4,
                         'secao': 5, 'subsecao': 6}
    stack = []
    article_id = None
    paragraph_id = None
    inciso_id = None
    annex_id = None
    records = []
    for index, (start, tipo, label) in enumerate(events):
        end = events[index + 1][0] if index + 1 < len(events) else len(text)
        record_id = hashlib.sha256(f'{caminho}|{start}|{tipo}'.encode('utf-8')).hexdigest()
        parent_id = None
        warning = ''
        if tipo == 'anexo':
            stack = []
            article_id = paragraph_id = inciso_id = None
            annex_id = record_id
        elif tipo in structural_levels:
            level = structural_levels[tipo]
            stack = [(rank, identifier) for rank, identifier in stack if rank < level]
            parent_id = stack[-1][1] if stack else annex_id
            stack.append((level, record_id))
            article_id = paragraph_id = inciso_id = None
        elif tipo == 'artigo':
            parent_id = stack[-1][1] if stack else annex_id
            article_id = record_id
            paragraph_id = inciso_id = None
        elif tipo == 'paragrafo':
            parent_id = article_id or annex_id
            paragraph_id = record_id
            inciso_id = None
            if article_id is None:
                warning = 'Parágrafo sem artigo ativo; revisar contexto.'
        elif tipo == 'inciso':
            parent_id = paragraph_id or article_id or annex_id
            inciso_id = record_id
            if not parent_id:
                warning = 'Inciso sem dispositivo superior reconhecido.'
        elif tipo == 'alinea':
            parent_id = inciso_id or paragraph_id or article_id or annex_id
            if inciso_id is None:
                warning = 'Alínea sem inciso ativo; revisar contexto.'

        original = text[start:end]
        cleaned = clean_legal_text(original)
        records.append({
            'record_id': record_id, 'parent_id': parent_id,
            'artigo_id': article_id, 'anexo_id': annex_id,
            'caminho_relativo': caminho, 'ordem': index + 1,
            'tipo_dispositivo': tipo, 'identificador_dispositivo': label,
            'posicao_inicio': start, 'posicao_fim': end,
            'texto_original': original, 'texto_estruturado': cleaned,
            'tamanho_caracteres': len(cleaned), 'tokens_estimados': len(cleaned) / 4,
            'normalizacao_aplicada': original != cleaned,
            'alerta_parser': warning, 'parser_version': PARSER_VERSION
        })
    return records


all_records = []
for _, row in df_fontes.iterrows():
    all_records.extend(parse_norm_to_structured_records(
        row['caminho_relativo'], source_texts[row['caminho_relativo']], row['body_start']
    ))
df_estruturado = pd.DataFrame(all_records)
if df_estruturado.empty:
    raise ValueError('O parser não gerou registros.')
df_estruturado = (
    df_estruturado.merge(df_fontes[['caminho_relativo', 'nome_arquivo', 'categoria', 'sha256_atual']],
                         on='caminho_relativo', how='left', validate='many_to_one')
    .merge(df_metadados.drop(columns='yaml_original'), on='caminho_relativo',
           how='left', validate='many_to_one')
)
display(df_estruturado.groupby('tipo_dispositivo').size().reset_index(name='quantidade'))

,tipo_dispositivo,quantidade
0,alinea,1285
1,anexo,3
2,artigo,5284
3,capitulo,387
4,inciso,4725
5,livro,19
6,paragrafo,5690
7,parte,4
8,preambulo,25
9,secao,481


## 5. Validação de Cobertura, Integridade e Pendências
Reconstruímos o texto original a partir do YAML e dos intervalos estruturados e verificamos a igualdade exata. Também conferimos identificadores, vínculos e quantidade de artigos em relação à EDA. Essa validação comprova preservação textual, não a correção semântica de todos os vínculos heurísticos. Pendências de metadados da auditoria permanecem disponíveis para revisão.

In [15]:
validation_records = []
for _, row in df_fontes.iterrows():
    caminho = row['caminho_relativo']
    records = df_estruturado[df_estruturado['caminho_relativo'] == caminho].sort_values('ordem')
    original = source_texts[caminho]
    rebuilt = original[:row['body_start']] + ''.join(records['texto_original'])
    contiguous = records.iloc[0]['posicao_inicio'] == row['body_start']
    contiguous = contiguous and records.iloc[-1]['posicao_fim'] == len(original)
    contiguous = contiguous and all(
        left == right for left, right in zip(records['posicao_fim'].iloc[:-1], records['posicao_inicio'].iloc[1:])
    )
    validation_records.append({'caminho_relativo': caminho,
        'preservacao_exata': rebuilt == original, 'intervalos_contiguos': contiguous,
        'artigos_parser': int(records['tipo_dispositivo'].eq('artigo').sum())})
df_validacao = pd.DataFrame(validation_records)
df_eda_normas = pd.read_csv(EDA_DIR / 'eda_normas.csv')
df_eda_normas['caminho'] = df_eda_normas['caminho'].str.replace('\\', '/', regex=False)
df_validacao = df_validacao.merge(df_eda_normas[['caminho', 'qtd_artigos']],
    left_on='caminho_relativo', right_on='caminho', how='left', validate='one_to_one')
df_validacao['artigos_conferem_eda'] = df_validacao['artigos_parser'].eq(df_validacao['qtd_artigos'])
if not df_validacao[['preservacao_exata', 'intervalos_contiguos', 'artigos_conferem_eda']].all().all():
    display(df_validacao)
    raise ValueError('Falha de preservação ou divergência com a EDA. Exportação interrompida.')
if not df_estruturado['record_id'].is_unique:
    raise ValueError('Identificadores de registros duplicados.')
known_ids = set(df_estruturado['record_id'])
if not df_estruturado['parent_id'].dropna().isin(known_ids).all():
    raise ValueError('Vínculo com registro superior inexistente.')

df_pendencias_parser = df_estruturado[df_estruturado['alerta_parser'] != ''].copy()
df_transformacoes = df_estruturado[df_estruturado['normalizacao_aplicada']][
    ['record_id', 'caminho_relativo', 'posicao_inicio', 'posicao_fim', 'texto_original', 'texto_estruturado']
].copy()
df_transformacoes['regra'] = 'Normalização de quebras de linha e remoção de espaços no fim das linhas.'
display(df_validacao)
print(f'• Normas com preservação exata confirmada: {len(df_validacao)}')
print(f'• Registros com alertas de hierarquia: {len(df_pendencias_parser)}')
print('• Revisar também amostras de vínculos sem alertas antes de considerar a estrutura definitiva.')

,caminho_relativo,preservacao_exata,intervalos_contiguos,artigos_parser,caminho,qtd_artigos,artigos_conferem_eda
0,codigos/codigo-civil-lei-10406-2002.md,True,True,2083,codigos/codigo-civil-lei-10406-2002.md,2083,True
1,codigos/codigo-de-processo-civil-lei-13105-201...,True,True,1074,codigos/codigo-de-processo-civil-lei-13105-201...,1074,True
2,codigos/consolidacao-das-leis-do-trabalho-clt-...,True,True,1016,codigos/consolidacao-das-leis-do-trabalho-clt-...,1016,True
3,constituicao/constituicao-1988.md,True,True,427,constituicao/constituicao-1988.md,427,True
4,leis/1949/lei-605-1949.md,True,True,18,leis/1949/lei-605-1949.md,18,True
5,leis/1962/lei-4090-1962.md,True,True,4,leis/1962/lei-4090-1962.md,4,True
6,leis/1965/lei-4749-1965.md,True,True,8,leis/1965/lei-4749-1965.md,8,True
7,leis/1970/lei-5584-1970.md,True,True,20,leis/1970/lei-5584-1970.md,20,True
8,leis/1973/lei-5889-1973.md,True,True,23,leis/1973/lei-5889-1973.md,23,True
9,leis/1974/lei-6019-1974.md,True,True,30,leis/1974/lei-6019-1974.md,30,True


• Normas com preservação exata confirmada: 25
• Registros com alertas de hierarquia: 535
• Revisar também amostras de vínculos sem alertas antes de considerar a estrutura definitiva.


## 6. Exportação da Base e dos Relatórios
CSV e JSON são exportados após as validações. Parquet é gerado quando o mecanismo opcional está disponível. A base é uma candidata para revisão e experimentação, com os alertas preservados; não é uma certificação jurídica. Nenhuma duplicidade ou marca jurídica é removida automaticamente.

In [16]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
tables = {
    'legislacao_estruturada': df_estruturado,
    'legislacao_metadados': df_metadados,
    'legislacao_fontes': df_fontes
}
for name, dataframe in tables.items():
    dataframe.to_csv(PROCESSED_DIR / f'{name}.csv', index=False, encoding='utf-8')
    dataframe.to_json(PROCESSED_DIR / f'{name}.json', orient='records', force_ascii=False, indent=4)
try:
    df_estruturado.to_parquet(PROCESSED_DIR / 'legislacao_estruturada.parquet', index=False)
    parquet_exportado = True
except ImportError:
    parquet_exportado = False
    print('• Parquet não gerado: instale pyarrow se esse formato for necessário. CSV e JSON estão disponíveis.')

reports = {
    'validacao_preservacao': df_validacao,
    'pendencias_parser': df_pendencias_parser,
    'transformacoes': df_transformacoes,
    'duplicacoes_decisoes': df_duplicacoes,
    'rastreabilidade': df_rastreabilidade,
    'pendencias_auditoria': df_audit
}
for name, dataframe in reports.items():
    dataframe.to_csv(REPORT_DIR / f'{name}.csv', index=False, encoding='utf-8')

resumo = {
    'executado_em_utc': EXECUTADO_EM, 'parser_version': PARSER_VERSION,
    'normas': len(df_fontes), 'registros': len(df_estruturado),
    'artigos': int(df_estruturado['tipo_dispositivo'].eq('artigo').sum()),
    'preservacao_textual_confirmada': True,
    'alertas_hierarquia': len(df_pendencias_parser),
    'parquet_exportado': parquet_exportado,
    'status': 'Base candidata com preservação textual validada; revisão semântica da hierarquia pendente.',
    'tokens': 'Estimativa caracteres / 4; usar tokenizador do modelo no notebook 04.'
}
with open(REPORT_DIR / 'resumo_estruturacao.json', 'w', encoding='utf-8') as f:
    json.dump(resumo, f, ensure_ascii=False, indent=4)
print(json.dumps(resumo, ensure_ascii=False, indent=4))
print(f'💾 Base: {PROCESSED_DIR}')
print(f'💾 Relatórios: {REPORT_DIR}')

{
    "executado_em_utc": "2026-11-05T03:34:19.097855+00:00",
    "parser_version": "1.0.0",
    "normas": 25,
    "registros": 18069,
    "artigos": 5284,
    "preservacao_textual_confirmada": true,
    "alertas_hierarquia": 535,
    "parquet_exportado": true,
    "status": "Base candidata com preservação textual validada; revisão semântica da hierarquia pendente.",
    "tokens": "Estimativa caracteres / 4; usar tokenizador do modelo no notebook 04."
}
💾 Base: C:\Users\LAISA\OneDrive\Documentos\lex_clt.ai\data\processed\legislacao
💾 Relatórios: C:\Users\LAISA\OneDrive\Documentos\lex_clt.ai\reports\legislacao\estruturacao


## 7. Contexto Hierárquico Legível
Percorremos os vínculos superiores para apresentar o contexto de cada dispositivo. Validamos ciclos e vínculos entre arquivos. O resultado facilita a inspeção, mas não aprova automaticamente a interpretação jurídica da hierarquia.

In [17]:
records_by_id = df_estruturado.set_index('record_id').to_dict(orient='index')

def get_ancestor_ids(record_id):
    ancestors = []
    visited = {record_id}
    current = records_by_id[record_id]
    parent_id = current['parent_id']
    while parent_id is not None and not pd.isna(parent_id):
        if parent_id in visited:
            raise ValueError(f'Ciclo na hierarquia: {record_id}')
        if parent_id not in records_by_id:
            raise ValueError(f'Registro superior inexistente: {parent_id}')
        parent = records_by_id[parent_id]
        if parent['caminho_relativo'] != current['caminho_relativo']:
            raise ValueError('Vínculo entre arquivos diferentes.')
        visited.add(parent_id)
        ancestors.append(parent_id)
        parent_id = parent['parent_id']
    return list(reversed(ancestors))


context_records = []
for record_id, row in records_by_id.items():
    chain = get_ancestor_ids(record_id) + [record_id]
    context = {'record_id': record_id}
    for tipo in ['parte', 'livro', 'titulo', 'capitulo', 'secao', 'subsecao', 'anexo', 'artigo']:
        labels = [records_by_id[item]['identificador_dispositivo'] for item in chain
                  if records_by_id[item]['tipo_dispositivo'] == tipo]
        context[f'contexto_{tipo}'] = labels[-1] if labels else None
    context['caminho_hierarquico'] = ' > '.join(
        records_by_id[item]['identificador_dispositivo'] for item in chain
    )
    context_records.append(context)

df_contexto = pd.DataFrame(context_records)
df_estruturado_contexto = df_estruturado.merge(df_contexto, on='record_id', validate='one_to_one')
display(df_estruturado_contexto[['caminho_relativo', 'tipo_dispositivo', 'caminho_hierarquico']].head(15))

,caminho_relativo,tipo_dispositivo,caminho_hierarquico
0,codigos/codigo-civil-lei-10406-2002.md,preambulo,Preâmbulo / estrutura inicial
1,codigos/codigo-civil-lei-10406-2002.md,parte,PARTE GERAL
2,codigos/codigo-civil-lei-10406-2002.md,livro,PARTE GERAL > LIVRO I — DAS PESSOAS
3,codigos/codigo-civil-lei-10406-2002.md,titulo,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...
4,codigos/codigo-civil-lei-10406-2002.md,capitulo,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...
5,codigos/codigo-civil-lei-10406-2002.md,artigo,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...
6,codigos/codigo-civil-lei-10406-2002.md,artigo,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...
7,codigos/codigo-civil-lei-10406-2002.md,artigo,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...
8,codigos/codigo-civil-lei-10406-2002.md,inciso,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...
9,codigos/codigo-civil-lei-10406-2002.md,inciso,PARTE GERAL > LIVRO I — DAS PESSOAS > TÍTULO I...


## 8. Revisão dos Alertas e Amostragem dos Vínculos
Agrupamos os alertas por motivo e norma e criamos uma fila persistente de decisões. Decisões existentes são preservadas ao reexecutar. Os campos de revisão são preenchidos após inspeção humana; nenhum alerta é marcado automaticamente como resolvido. Amostras sem alerta também precisam de inspeção.

In [18]:
df_alertas_resumo = (
    df_pendencias_parser.groupby(['caminho_relativo', 'alerta_parser'])
    .size().reset_index(name='quantidade')
    .sort_values('quantidade', ascending=False)
)
display(df_alertas_resumo)
display(df_estruturado_contexto[df_estruturado_contexto['alerta_parser'] != '']
        .groupby('alerta_parser', sort=True).head(3)[
            ['record_id', 'caminho_relativo', 'caminho_hierarquico', 'alerta_parser', 'texto_original']])

samples = []
for tipo, group in df_estruturado_contexto[
    df_estruturado_contexto['alerta_parser'] == ''
].groupby('tipo_dispositivo', sort=True):
    samples.append(group.sample(n=min(5, len(group)), random_state=42))
df_amostras_vinculos = pd.concat(samples, ignore_index=True) if samples else df_estruturado_contexto.iloc[:0].copy()
display(df_amostras_vinculos[['record_id', 'caminho_relativo', 'caminho_hierarquico', 'texto_original']])

review_columns = ['record_id', 'caminho_relativo', 'tipo_dispositivo', 'identificador_dispositivo',
                  'parent_id', 'caminho_hierarquico', 'alerta_parser', 'texto_original']
df_fila_revisao = pd.concat([
    df_estruturado_contexto[df_estruturado_contexto['alerta_parser'] != ''][review_columns],
    df_amostras_vinculos[review_columns]
]).drop_duplicates('record_id')
decision_path = REPORT_DIR / 'decisoes_revisao.csv'
decision_fields = ['decisao', 'parent_id_proposto', 'justificativa', 'revisor', 'revisado_em']
if decision_path.is_file():
    previous = pd.read_csv(decision_path, dtype=str, keep_default_na=False)
    if previous['record_id'].duplicated().any():
        raise ValueError('Decisões com record_id duplicado; revise o arquivo de decisões.')
    for field in decision_fields:
        if field not in previous:
            previous[field] = ''
    # Arquiva a versão anterior para preservar decisões de registros fora da fila atual.
    previous.to_csv(REPORT_DIR / 'decisoes_revisao_anteriores.csv', index=False, encoding='utf-8')
    df_fila_revisao = df_fila_revisao.merge(previous[['record_id', *decision_fields]],
                                          on='record_id', how='left', validate='one_to_one')
else:
    for field in decision_fields:
        df_fila_revisao[field] = ''
df_fila_revisao[decision_fields] = df_fila_revisao[decision_fields].fillna('')
df_fila_revisao.to_csv(decision_path, index=False, encoding='utf-8')
print(f'• Fila de revisão: {len(df_fila_revisao)} registros. Decisões propostas não alteram a base automaticamente.')

,caminho_relativo,alerta_parser,quantidade
0,codigos/consolidacao-das-leis-do-trabalho-clt-...,Alínea sem inciso ativo; revisar contexto.,388
8,leis/1991/lei-8212-1991.md,Alínea sem inciso ativo; revisar contexto.,81
9,leis/1991/lei-8213-1991.md,Alínea sem inciso ativo; revisar contexto.,15
5,leis/1974/lei-6019-1974.md,Alínea sem inciso ativo; revisar contexto.,14
3,leis/1949/lei-605-1949.md,Alínea sem inciso ativo; revisar contexto.,13
7,leis/1990/lei-8036-1990.md,Alínea sem inciso ativo; revisar contexto.,12
1,codigos/consolidacao-das-leis-do-trabalho-clt-...,Inciso sem dispositivo superior reconhecido.,3
2,codigos/consolidacao-das-leis-do-trabalho-clt-...,Parágrafo sem artigo ativo; revisar contexto.,3
4,leis/1973/lei-5889-1973.md,Alínea sem inciso ativo; revisar contexto.,3
6,leis/1985/lei-7418-1985.md,Alínea sem inciso ativo; revisar contexto.,3


,record_id,caminho_relativo,caminho_hierarquico,alerta_parser,texto_original
8487,02ffb6f275f03348cdb3496866a142c3f9b84a90710abc...,codigos/consolidacao-das-leis-do-trabalho-clt-...,TÍTULO I — INTRODUÇÃO > Art. 7º > a,Alínea sem inciso ativo; revisar contexto.,"a) aos empregados domésticos, assim considerad..."
8488,6e9f39c842ff8bec57b83bec12aeb5ab06087cf50579bd...,codigos/consolidacao-das-leis-do-trabalho-clt-...,TÍTULO I — INTRODUÇÃO > Art. 7º > b,Alínea sem inciso ativo; revisar contexto.,"b) aos trabalhadores rurais, assim considerado..."
8489,b56e743a8785c5191ea5b25cada03dcefa22412cc4c691...,codigos/consolidacao-das-leis-do-trabalho-clt-...,TÍTULO I — INTRODUÇÃO > Art. 7º > c,Alínea sem inciso ativo; revisar contexto.,"c) aos funcionários públicos da União, dos Est..."
8951,36135574d526cfbf98c8d62229bf34b57ac9c495a7a347...,codigos/consolidacao-das-leis-do-trabalho-clt-...,I,Inciso sem dispositivo superior reconhecido.,I - as precauções de segurança na movimentação...
8952,0be5a9249c4b3f0aa698236280006a33a1faff0fd40fde...,codigos/consolidacao-das-leis-do-trabalho-clt-...,II,Inciso sem dispositivo superior reconhecido.,II - as exigências similares relativas ao manu...
8953,703e35ca9d7685535a608486095db46d8562baf155912b...,codigos/consolidacao-das-leis-do-trabalho-clt-...,III,Inciso sem dispositivo superior reconhecido.,III - a obrigatoriedade de indicação de carga ...
8954,6e79949deb27679a5c39197073506a0f564fa75fb5ca2c...,codigos/consolidacao-das-leis-do-trabalho-clt-...,Parágrafo único,Parágrafo sem artigo ativo; revisar contexto.,Parágrafo único - As disposições relativas ao ...
8967,2d9c2b80e972a730ac17d44fc69b5bdec0fed446e17d5c...,codigos/consolidacao-das-leis-do-trabalho-clt-...,Parágrafo único,Parágrafo sem artigo ativo; revisar contexto.,Parágrafo único - As normas referidas neste ar...
8989,705d8b4b8bb86769dadb9eef74428815f8bda7d82fee60...,codigos/consolidacao-das-leis-do-trabalho-clt-...,Parágrafo único,Parágrafo sem artigo ativo; revisar contexto.,Parágrafo único - Não está compreendida na pro...


,record_id,caminho_relativo,caminho_hierarquico,texto_original
0,f347b29c94e7c47071798a92e77f23f3222f17b7659d8a...,constituicao/constituicao-1988.md,TÍTULO VI — Da Tributação e do Orçamento > CAP...,c) as regras de distribuição aplicáveis aos re...
1,c17dcd81783f6d9f99f3caf661e3e4ffdb09e79c60a805...,codigos/consolidacao-das-leis-do-trabalho-clt-...,TÍTULO V — DA ORGANIZAÇÃO SINDICAL > CAPÍTULO ...,l) prevenção de acidentes do trabalho; (Incluí...
2,a0b7bbe4c6f647b66475455561a7ff1cf271a5d7297f77...,constituicao/constituicao-1988.md,TÍTULO VI — Da Tributação e do Orçamento > CAP...,b) de representantes dos Estados e do Distrito...
3,1fe4f902380e10da7d074dae24e36befaf9eb060736668...,leis/1991/lei-8212-1991.md,TÍTULO VI — DO FINANCIAMENTO DA SEGURIDADE SOC...,a) aquele que presta serviço de natureza urban...
4,c43d15a8d064843b723de3dd19a9ccd95a85499f56d969...,leis/1991/lei-8212-1991.md,TÍTULO VI — DO FINANCIAMENTO DA SEGURIDADE SOC...,"b) pescador artesanal ou a este assemelhado, q..."
5,1e216935845de10bebfb618709950cbafcf01ddd72996c...,codigos/consolidacao-das-leis-do-trabalho-clt-...,ANEXO — Quadro a que se refere o art. 577 da C...,## ANEXO — Quadro a que se refere o art. 577 d...
6,0bc8d51cb5f40c69faefd707b4739a63ed220840082053...,leis/1990/lei-8036-1990.md,ANEXO,"## ANEXO\n\n(Incluído pela Lei nº 13.932, de 2..."
7,23601719b884894898710c67b47bfcfc99eded1a998843...,leis/2000/lei-10101-2000.md,ANEXO,"## ANEXO\n\n(Incluído pela Lei nº 12.832, de 2..."
8,2b42edbc0f7ef9f07440ad03a22e3caf32a7eed7cf1ee1...,codigos/codigo-de-processo-civil-lei-13105-201...,PARTE ESPECIAL > LIVRO I — DO PROCESSO DE CONH...,**Art. 503.** A decisão que julgar total ou pa...
9,3ad845dea5f625ba02c4a611f2f0494464ef74ebcb2595...,codigos/consolidacao-das-leis-do-trabalho-clt-...,Art. 2º,##### Art. 2º\n\nO presente decreto-lei entrar...


• Fila de revisão: 592 registros. Decisões propostas não alteram a base automaticamente.


## 9. Conferência de Títulos, Anexos e Limites dos Artigos
Verificamos se títulos e anexos foram separados e se o anexo encerrou o contexto do artigo anterior. Esses controles estruturais não substituem a revisão dos exemplos nem confirmam que todos os formatos possíveis foram reconhecidos.

In [20]:
structure_types = ['parte', 'livro', 'titulo', 'capitulo', 'secao', 'subsecao', 'anexo']
df_estruturas = df_estruturado_contexto[
    df_estruturado_contexto['tipo_dispositivo'].isin(structure_types)
].copy()
df_estruturas['artigo_ativo_indevido'] = df_estruturas['artigo_id'].notna()
if df_estruturas['artigo_ativo_indevido'].any():
    raise ValueError('Título ou anexo associado indevidamente ao contexto de um artigo.')

boundary_records = []
for _, row in df_estruturas.iterrows():
    same_file = df_estruturado[df_estruturado['caminho_relativo'] == row['caminho_relativo']]
    previous = same_file[same_file['ordem'] == row['ordem'] - 1]
    boundary_records.append({
        'record_id': row['record_id'], 'caminho_relativo': row['caminho_relativo'],
        'tipo_dispositivo': row['tipo_dispositivo'],
        'identificador_dispositivo': row['identificador_dispositivo'],
        'limite_anterior_ok': previous.empty or int(previous.iloc[0]['posicao_fim']) == row['posicao_inicio'],
        'texto_amostra': row['texto_original'][:500]
    })
df_validacao_estruturas = pd.DataFrame(boundary_records)
if not df_validacao_estruturas.empty and not df_validacao_estruturas['limite_anterior_ok'].all():
    raise ValueError('Limites estruturais inconsistentes.')
display(df_validacao_estruturas.groupby('tipo_dispositivo', sort=True).head(3))
print(f'• Títulos e anexos separados: {len(df_estruturas)}')

,record_id,caminho_relativo,tipo_dispositivo,identificador_dispositivo,limite_anterior_ok,texto_amostra
0,b97935ec3c598873f1b3d83bb7d43b75e718bda1b99de6...,codigos/codigo-civil-lei-10406-2002.md,parte,PARTE GERAL,True,## PARTE GERAL\n\n
1,a1d4436a3d7571db901945cc2ee58551fc2b2cf83a734e...,codigos/codigo-civil-lei-10406-2002.md,livro,LIVRO I — DAS PESSOAS,True,### LIVRO I — DAS PESSOAS\n\n
2,4950f79bf7ed877ab2270834953177e708fc59823a7524...,codigos/codigo-civil-lei-10406-2002.md,titulo,TÍTULO I — DAS PESSOAS NATURAIS,True,#### TÍTULO I — DAS PESSOAS NATURAIS\n\n
3,8e5b01d81715dfc17a2789e92e9abe21a72706428f62da...,codigos/codigo-civil-lei-10406-2002.md,capitulo,CAPÍTULO I — Da Personalidade e da Capacidade,True,##### CAPÍTULO I — Da Personalidade e da Capac...
4,9657489263d388357823abee49ed19b75121b6caed2a8d...,codigos/codigo-civil-lei-10406-2002.md,capitulo,CAPÍTULO II — Dos Direitos da Personalidade,True,##### CAPÍTULO II — Dos Direitos da Personalid...
5,2f83363384771df9290701a684e6c3621fef7f5484b74a...,codigos/codigo-civil-lei-10406-2002.md,capitulo,CAPÍTULO III — Da Ausência,True,##### CAPÍTULO III — Da Ausência\n\n
6,4d66d6df738da93ee2fea2bc5dc57decfd4cfaeeaf537b...,codigos/codigo-civil-lei-10406-2002.md,secao,Seção I — Da Curadoria dos Bens do Ausente,True,###### Seção I — Da Curadoria dos Bens do Ause...
7,84d548aaf1baea863da5a19657eac559a421b72b762831...,codigos/codigo-civil-lei-10406-2002.md,secao,Seção II — Da Sucessão Provisória,True,###### Seção II — Da Sucessão Provisória\n\n
8,a4272655ecfd84ac1170ed5f54ab76d17f40af0b501f3c...,codigos/codigo-civil-lei-10406-2002.md,secao,Seção III — Da Sucessão Definitiva,True,###### Seção III — Da Sucessão Definitiva\n\n
9,1a1a66fecab3f988699e336c4f5478d5e6e8521d7fad41...,codigos/codigo-civil-lei-10406-2002.md,titulo,TÍTULO II — DAS PESSOAS JURÍDICAS,True,#### TÍTULO II — DAS PESSOAS JURÍDICAS\n\n


• Títulos e anexos separados: 1060


## 10. Reconstrução de Artigos Completos para o Chunking
Cada artigo é reunido aos seus descendentes na ordem original. Preservamos notas e referências presentes nesses intervalos. Exigimos que os descendentes coincidam exatamente com o intervalo do artigo até o próximo artigo ou título estrutural. Artigos com alertas permanecem sinalizados para revisão.

In [21]:
article_records = []
for caminho, group in df_estruturado_contexto.groupby('caminho_relativo', sort=True):
    group = group.sort_values('ordem')
    for _, article in group[group['tipo_dispositivo'] == 'artigo'].iterrows():
        identifier = article['record_id']
        descendants = group[group['artigo_id'] == identifier].sort_values('ordem')
        if not all(item == identifier or identifier in get_ancestor_ids(item) for item in descendants['record_id']):
            raise ValueError('Contexto de artigo incompatível com os vínculos superiores.')
        after = group[group['ordem'] > article['ordem']]
        stop = after[after['tipo_dispositivo'].isin(['artigo', *structure_types])]
        end = int(stop.iloc[0]['posicao_inicio']) if not stop.empty else len(source_texts[caminho])
        expected = source_texts[caminho][article['posicao_inicio']:end]
        rebuilt = ''.join(descendants['texto_original'])
        if rebuilt != expected:
            raise ValueError(f'Descendentes não cobrem o artigo: {caminho} | {article["identificador_dispositivo"]}')
        normalized = clean_legal_text(rebuilt)
        article_records.append({
            'artigo_id': identifier, 'caminho_relativo': caminho,
            'identificador_artigo': article['identificador_dispositivo'],
            'caminho_hierarquico': article['caminho_hierarquico'],
            'posicao_inicio': article['posicao_inicio'], 'posicao_fim': end,
            'quantidade_registros': len(descendants),
            'record_ids_json': json.dumps(descendants['record_id'].tolist()),
            'texto_original': rebuilt, 'texto_artigo_completo': normalized,
            'tokens_estimados': len(normalized) / 4,
            'tem_alerta_parser': bool(descendants['alerta_parser'].ne('').any()),
            'sha256_fonte': article['sha256_atual'],
            'status_declarado': article['status_declarado'], 'source': article['source'],
            'date': article['date'], 'retrieved': article['retrieved']
        })
df_artigos_completos = pd.DataFrame(article_records)
assert len(df_artigos_completos) == df_estruturado['tipo_dispositivo'].eq('artigo').sum()
print(f'• Artigos reconstruídos e conferidos por intervalo: {len(df_artigos_completos)}')
print(f"• Artigos contendo alertas do parser: {df_artigos_completos['tem_alerta_parser'].sum()}")
display(df_artigos_completos[['caminho_relativo', 'identificador_artigo', 'quantidade_registros', 'tokens_estimados']].head(10))

• Artigos reconstruídos e conferidos por intervalo: 5284
• Artigos contendo alertas do parser: 118


,caminho_relativo,identificador_artigo,quantidade_registros,tokens_estimados
0,codigos/codigo-civil-lei-10406-2002.md,Art. 1º,1,17.75
1,codigos/codigo-civil-lei-10406-2002.md,Art. 2º,1,36.00
2,codigos/codigo-civil-lei-10406-2002.md,Art. 3º,4,98.75
3,codigos/codigo-civil-lei-10406-2002.md,Art. 4º,6,150.25
4,codigos/codigo-civil-lei-10406-2002.md,Art. 5º,7,180.00
5,codigos/codigo-civil-lei-10406-2002.md,Art. 6º,1,42.50
6,codigos/codigo-civil-lei-10406-2002.md,Art. 7º,4,118.75
7,codigos/codigo-civil-lei-10406-2002.md,Art. 8º,1,44.50
8,codigos/codigo-civil-lei-10406-2002.md,Art. 9º,5,70.75
9,codigos/codigo-civil-lei-10406-2002.md,Art. 10,4,83.50


## 11. Casos de Validação e Exportação Complementar
Validamos formatos específicos com exemplos sintéticos e exportamos as evidências de revisão. As verificações sintéticas cobrem comportamentos esperados, sem substituir a inspeção das fontes reais. O estado de revisão permanece pendente até as decisões serem analisadas e aplicadas.

In [22]:
validation_cases = [
    ('Artigo com sufixo', '## Art. 10-A\nTexto.\n', ['artigo']),
    ('Dispositivo vetado', '**Art. 7º**\n(VETADO).\n', ['artigo']),
    ('Hierarquia completa', '##### Art. 1º\nTexto.\nParágrafo único. Condição.\nI - Item\na) Alínea\n',
     ['artigo', 'paragrafo', 'inciso', 'alinea']),
    ('Anexo com tabela', '##### Art. 1º\nTexto.\n## ANEXO\n<table>\nI - texto tabular\n</table>\n',
     ['artigo', 'anexo'])
]
case_results = []
for name, text, expected_types in validation_cases:
    parsed = parse_norm_to_structured_records(f'validacao/{name}', text, 0)
    actual_types = [item['tipo_dispositivo'] for item in parsed]
    assert actual_types == expected_types, name
    assert ''.join(item['texto_original'] for item in parsed) == text, name
    if name == 'Hierarquia completa':
        assert parsed[2]['parent_id'] == parsed[1]['record_id']
        assert parsed[3]['parent_id'] == parsed[2]['record_id']
    case_results.append({'caso': name, 'resultado': 'OK'})
df_casos_validacao = pd.DataFrame(case_results)
display(df_casos_validacao)

for name, dataframe in {
    'legislacao_estruturada_contexto': df_estruturado_contexto,
    'legislacao_artigos_completos': df_artigos_completos
}.items():
    dataframe.to_csv(PROCESSED_DIR / f'{name}.csv', index=False, encoding='utf-8')
    dataframe.to_json(PROCESSED_DIR / f'{name}.json', orient='records', force_ascii=False, indent=4)
    if parquet_exportado:
        dataframe.to_parquet(PROCESSED_DIR / f'{name}.parquet', index=False)

for name, dataframe in {
    'alertas_por_norma_motivo': df_alertas_resumo,
    'amostras_vinculos_sem_alerta': df_amostras_vinculos,
    'validacao_titulos_anexos': df_validacao_estruturas,
    'casos_validacao_parser': df_casos_validacao
}.items():
    dataframe.to_csv(REPORT_DIR / f'{name}.csv', index=False, encoding='utf-8')

resumo.update({
    'artigos_completos_reconstruidos': len(df_artigos_completos),
    'artigos_com_alertas': int(df_artigos_completos['tem_alerta_parser'].sum()),
    'registros_fila_revisao': len(df_fila_revisao),
    'revisao_humana': 'Pendente; decisões propostas não são aplicadas automaticamente.'
})
with open(REPORT_DIR / 'resumo_estruturacao.json', 'w', encoding='utf-8') as f:
    json.dump(resumo, f, ensure_ascii=False, indent=4)
print('✅ Etapas complementares concluídas. Consulte a fila de revisão antes de aprovar a hierarquia.')

,caso,resultado
0,Artigo com sufixo,OK
1,Dispositivo vetado,OK
2,Hierarquia completa,OK
3,Anexo com tabela,OK


✅ Etapas complementares concluídas. Consulte a fila de revisão antes de aprovar a hierarquia.
